# CT Scans Report Generation (Training and Testing) — Cleaned Version

This notebook is a cleaned-up version of my workflow for generating CT scan reports.

Multiple experiments were conducted, including testing different model architectures, hyperparameters, and data preprocessing techniques. For clarity and readability, all trial-and-error code, intermediate results, and unnecessary steps have been removed.

This version contains only the essential code needed to reproduce the process of training the model and generating reports for CT scans, within the given computational constraints.

**NOTE:** CT scans are very complex and large, often around 400 MB each, which makes processing and report generation challenging. With limited computational resources, I did my best to train and evaluate the model on this demanding data.



Mount Google Drive, Hugging Face login

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from huggingface_hub import login
login(token="#")


Install dependencies

In [ ]:
!pip install --quiet \
  transformers bitsandbytes accelerate peft datasets \
  nibabel torch torchvision tqdm huggingface_hub


Imports & Accelerator

In [ ]:
 import os, torch, nibabel as nib
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model
from accelerate import Accelerator

# Initialize Accelerator
accel = Accelerator(
    mixed_precision="fp16",
    gradient_accumulation_steps=4
)
device = accel.device
print("Accelerator device:", device)


Vision model and LoRA base

In [ ]:
from huggingface_hub import hf_hub_download
import os
import shutil

repo_id = "ibrahimhamamci/CT-RATE"
repo_type = "dataset"
base_local_dir = "ct_weights"

files_to_download = [
    "models/CT-CLIP-Related/CT-CLIP_v2.pt",
    "models/CT-CHAT/llama_3.1_8b/adapter_config.json",
    "models/CT-CHAT/llama_3.1_8b/adapter_model.safetensors",
]
for file_path in files_to_download:
    # Download into a temporary directory
    temp_path = hf_hub_download(
        repo_id=repo_id,
        filename=file_path,
        repo_type=repo_type,
        local_dir="temp_hf_download",
        local_dir_use_symlinks=False
    )

    # Reconstruct original folder structure in base_local_dir
    full_dest_path = os.path.join(base_local_dir, file_path)
    os.makedirs(os.path.dirname(full_dest_path), exist_ok=True)
    shutil.move(temp_path, full_dest_path)

    print(f"Downloaded and moved: {full_dest_path}")

In [ ]:
!git clone https://github.com/ibrahimethemhamamci/CT-CLIP.git /content/CT-CLIP
import sys, os
paths = [
    "/content/CT-CLIP",                    # for CT_CLIP.ctvit
    "/content/CT-CLIP/transformer_maskgit",# transformer lib
]
for p in paths:
    if not os.path.isdir(p):
        raise FileNotFoundError(f"Missing path: {p}")
sys.path.extend(paths)
print("Paths set:", paths)


In [ ]:
%cd /content/CT-CLIP/transformer_maskgit
!pip install -e .
%cd /content

Initialize and Load CTViT Model

In [ ]:
from transformer_maskgit.ctvit import CTViT
import torch

# Instantiate on CPU
ct_vit = CTViT(
    dim=512, codebook_size=8192, image_size=480,
    patch_size=24, temporal_patch_size=12,
    spatial_depth=4, temporal_depth=4,
    dim_head=32, heads=8, channels=1,
    use_vgg_and_gan=False
).eval().cpu()

# Load checkpoint (map to CPU first)
sd = torch.load("/content/ct_weights/models/CT-CLIP-Related/CT-CLIP_v2.pt", map_location="cpu")
sd = sd.get("state_dict", sd)
sd = {k.replace("module.",""):v for k,v in sd.items()}
ct_vit.load_state_dict(sd, strict=False)

# Move to GPU
ct_vit = ct_vit.cuda()
print(" CTViT ready on", next(ct_vit.parameters()).device)


**Function to Extract Features from CT Volumes Using CTViT**

-Loads a 3D CT scan volume using nibabel.

-Normalizes intensity values to a 0-1 range (clamping and scaling).

-Adds batch and channel dimensions, moves data to GPU.

-Resizes volume to fixed spatial and slice dimensions with trilinear interpolation.

-Passes the volume through the CTViT model to get encoded tokens (features).

-Reshapes and flattens tokens, then moves them back to CPU.

-Returns the extracted features as a tensor.

In [ ]:
import nibabel as nib
import torch
import torch.nn.functional as F

def extract_ctvit_features(ct_vit_model, volume_path,
                           max_slices=12, size=480):
    # Load & normalize on CPU
    vol = nib.load(volume_path).get_fdata().astype("float32")
    vol = torch.clamp(torch.from_numpy(vol), -1000.0, 400.0)
    vol = (vol + 1000.0) / 1400.0

    # To [1,1,D,H,W] → GPU
    x = vol.unsqueeze(0).unsqueeze(0).cuda()
    x = F.interpolate(x, size=(max_slices, size, size),
                      mode="trilinear", align_corners=False)

    # Forward through CTViT
    with torch.no_grad():
        tokens = ct_vit_model(x, return_encoded_tokens=True)

    # Flatten → CPU
    B, T, H, W, D = tokens.shape
    feats = tokens.reshape(B, T*H*W, D).squeeze(0).cpu()
    return feats


Simple projection head


In [ ]:
import torch.nn as nn

class ProjectionHead(nn.Module):
    def __init__(self, in_dim=512, out_dim=4096):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(inplace=True),
            nn.Linear(in_dim, out_dim),
        )
    def forward(self, x):
        pooled = x.mean(dim=1)
        return self.net(pooled)

proj_head = ProjectionHead(in_dim=512, out_dim=4096).to(device)
state = torch.load("/content/drive/MyDrive/projection_head_100vols.pt",
                   map_location=device)
proj_head.load_state_dict(state)
proj_head.eval()
print("Loaded 100-scan projection head")


Download and Prepare CT Volume Dataset

In [ ]:
!git clone --quiet https://github.com/sezginerr/example_download_script.git
%cd example_download_script

from huggingface_hub import hf_hub_download
import pandas as pd, os, torch, nibabel as nib
import torch.nn.functional as F

repo_id        = "ibrahimhamamci/CT-RATE"
directory_name = "dataset/train/"
data           = pd.read_csv("train_labels.csv")
max_scans      = 200
download_dir   = "data_volumes"

os.makedirs(download_dir, exist_ok=True)
count = 0
for name in data["VolumeName"]:
    f1, f2, f3 = name.split("_")[:3]
    subfolder  = f"{directory_name}{f1}_{f2}/{f1}_{f2}_{f3}"
    hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        subfolder=subfolder,
        filename=name,
        local_dir=download_dir,
        token="#"
    )
    count += 1
    if count >= max_scans:
        break
print(f"Downloaded {count} volumes to {download_dir}")
%cd ..

In [ ]:
%cd content

Recursive feature extraction

In [ ]:
data_root   = "example_download_script/data_volumes"
feats_list  = []
names_list  = []

for dirpath, _, files in os.walk(data_root):
    for fn in files:
        if not fn.endswith(".nii.gz"):
            continue
        path = os.path.join(dirpath, fn)

        # preprocess volume
        vol = nib.load(path).get_fdata().astype("float32")
        vol = torch.clamp(torch.from_numpy(vol), -1000.0, 400.0)
        vol = (vol + 1000.0) / 1400.0
        x   = vol.unsqueeze(0).unsqueeze(0).cuda()
        x   = F.interpolate(x, size=(12,480,480),
                            mode="trilinear", align_corners=False)

        # forward pass
        with torch.no_grad():
            tokens = ct_vit(x, return_encoded_tokens=True)  # [1,T,H,W,D]
        B, T, H, W, D = tokens.shape
        feats = tokens.reshape(B, T*H*W, D).squeeze(0).cpu()  # [N_patches,512]

        # sanity
        assert not feats.isnan().any(), f"NaN in {fn}"
        assert not feats.isinf().any(), f"Inf in {fn}"
        print(f" {fn} → {feats.shape}")

        feats_list.append(feats)
        names_list.append(fn[:-7])

print(f"Extracted features for {len(feats_list)} volumes")

Download and Process Radiology Reports CSV; Build Lookup Dictionary for Reports

In [ ]:
import pandas as pd
import os
from huggingface_hub import hf_hub_download

# Download the train_reports.csv
reports_path = hf_hub_download(
    repo_id="ibrahimhamamci/CT-RATE",
    repo_type="dataset",
    subfolder="dataset/radiology_text_reports",
    filename="train_reports.csv",
    use_auth_token=True
)
print("CSV downloaded to:", reports_path)

# Load and drop any empty entries
df_reports = pd.read_csv(reports_path).dropna(subset=["Findings_EN", "Impressions_EN"])
print(f"Loaded {len(df_reports)} non‑empty report rows")

# Build lookup dict, stripping “.nii.gz”
report_index = {}
for _, row in df_reports.iterrows():
    name = row.VolumeName
    if name.endswith(".nii.gz"):
        key = name[:-7]
    else:
        key = name
    report_index[key] = {
        "Findings_EN": row.Findings_EN,
        "Impressions_EN": row.Impressions_EN
    }

# Sanity‑check alignment with extracted names_list
missing = [n for n in names_list if n not in report_index]
assert len(missing)==0, f"Reports missing for: {missing[:5]}… ({len(missing)} total)"
print(f" report_index contains all {len(names_list)} extracted volumes")


Dataset Class for Combining CT Scan Features and Radiology Text Reports with Tokenizer Embeddings

In [ ]:
class CTChatDataset(Dataset):
    def __init__(self, names_list, feats_list, report_index, tokenizer, llm_emb):
        self.names = names_list
        self.feats = feats_list
        self.report_index = report_index
        self.tokenizer = tokenizer
        self.llm_emb = llm_emb  # model.get_input_embeddings()
    def __len__(self):
        return len(self.names)
    def __getitem__(self, idx):
        name  = self.names[idx]
        feats = self.feats[idx].to(device).unsqueeze(0)  # [1,N,512]
        proj  = proj_head(feats).squeeze(0)              # [4096]
        text  = self.report_index[name]["Findings_EN"] + "\n" + self.report_index[name]["Impressions_EN"]
        toks  = self.tokenizer(text, return_tensors="pt",
                               padding="longest", truncation=True, max_length=512)
        input_ids = toks.input_ids.to(device)
        attn_mask = toks.attention_mask.to(device)
        # text embeddings target
        text_embs = self.llm_emb(input_ids)             # [1, L, dim]
        proj_emb  = proj.unsqueeze(0).unsqueeze(1).to(text_embs.dtype)  # [1,1,dim]
        inputs_embeds = torch.cat([proj_emb, text_embs], dim=1)         # [1,L+1,dim]
        amask = torch.cat([torch.ones(1,1,device=device), attn_mask], dim=1)
        labels = torch.cat([torch.full((1,1), -100, device=device), input_ids], dim=1)
        return {
            "inputs_embeds": inputs_embeds.squeeze(0),
            "attention_mask": amask.squeeze(0),
            "labels": labels.squeeze(0),
        }


In [ ]:
import gc, torch
try:
    del base, model
except:
    pass
gc.collect()
torch.cuda.empty_cache()
print(" Cleared old model from GPU")


Initialize 8-bit Quantized LLaMA Model + Tokenizer + LoRA (PEFT) for Efficient Fine-Tuning

In [ ]:
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM
from peft       import LoraConfig, get_peft_model

# Pure 8-bit config (no offload)
bnb = BitsAndBytesConfig(load_in_8bit=True, llm_int8_enable_fp32_cpu_offload=False)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    trust_remote_code=True, padding_side="left"
)
tokenizer.pad_token_id = tokenizer.eos_token_id

# Force everything onto GPU 0
device_map = {"": 0}

base = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=bnb,
    device_map=device_map,
    trust_remote_code=True
)

# Wrap in LoRA exactly as before
lora_cfg = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(base, lora_cfg)
model.gradient_checkpointing_enable()

# Re-prepare with the same Accelerator & objects
model, proj_head, optimizer, train_dl, scheduler = accel.prepare(
    model, proj_head, optimizer, train_dl, scheduler
)

print("loaded 8-bit model + LoRA on GPU")


In [ ]:
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

# Tokenizer & Base Model with CPU offload
bnb = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=False
)
tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    padding_side="left", trust_remote_code=True
)
tokenizer.pad_token_id = tokenizer.eos_token_id

base = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True
)

# LoRA Config
lora_cfg = LoraConfig(
    r=4,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(base, lora_cfg)
model.gradient_checkpointing_enable()

# Wrap for Accelerator
model = accel.prepare(model)

# Embedding layer for dataset
llm_emb = base.get_input_embeddings()


Dataset & Dataloader Setup + Optimizer and Scheduler Configuration

In [ ]:
# Instantiate dataset & dataloader
llm_emb = base.get_input_embeddings()
dataset = CTChatDataset(names_list, feats_list, report_index, tokenizer, llm_emb)
train_dl = DataLoader(dataset, batch_size=1, shuffle=True)

train_dl = accel.prepare(train_dl)

# Optimizer & LR scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
num_steps = len(train_dl) * 3  # e.g. 3 epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=10, num_training_steps=num_steps
)

model, optimizer, scheduler = accel.prepare(model, optimizer, scheduler)


Fine-Tuning LoRA + Projection Head Jointly on CT Features and Reports

In [ ]:
# LoRA Config with larger rank
lora_cfg = LoraConfig(
    r=16,           # ↑ double from 4
    lora_alpha=32,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(base, lora_cfg)
model.gradient_checkpointing_enable()

# Build dataset & dataloader as before
dataset = CTChatDataset(names_list, feats_list, report_index, tokenizer, lambda ids: model.get_input_embeddings()(ids))
train_dl = DataLoader(dataset, batch_size=1, shuffle=True)

# Optimizer now includes proj_head
optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(proj_head.parameters()),
    lr=5e-5
)
num_steps = len(train_dl) * 8   # e.g. 8 epochs
scheduler = get_linear_schedule_with_warmup(optimizer, 10, num_steps)

# Single prepare call
model, proj_head, optimizer, train_dl, scheduler = accel.prepare(
    model, proj_head, optimizer, train_dl, scheduler
)
best_val = float("inf")
patience = 2
wait = 0
# Training loop
model.train(); proj_head.train()
for epoch in range(8):
    total_loss = 0.0
    for batch in train_dl:
        outputs = model(
            inputs_embeds=batch["inputs_embeds"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )
        loss = outputs.loss / accel.gradient_accumulation_steps
        accel.backward(loss)
        if accel.sync_gradients:
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} avg loss {total_loss/len(train_dl):.4f}")
    #simple early stopping
    avg_loss = total_loss/len(train_dl)
    if avg_loss < best_val:
        best_val = avg_loss
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print("Stopping early at epoch", epoch+1)
            break

# Saving both
model.save_pretrained("/content/drive/MyDrive/ct_lora_500scans")
torch.save(proj_head.state_dict(), "/content/drive/MyDrive/projection_head_500scans_joint.pt")


# **TESTING**

Load CTViT

In [ ]:
from transformer_maskgit.ctvit import CTViT
import torch

# Instantiate on CPU
ct_vit = CTViT(
    dim=512, codebook_size=8192, image_size=480,
    patch_size=24, temporal_patch_size=12,
    spatial_depth=4, temporal_depth=4,
    dim_head=32, heads=8, channels=1,
    use_vgg_and_gan=False
).eval().cpu()

# Load checkpoint (map to CPU first)
sd = torch.load("/content/ct_weights/models/CT-CLIP-Related/CT-CLIP_v2.pt", map_location="cpu")
sd = sd.get("state_dict", sd)
sd = {k.replace("module.",""):v for k,v in sd.items()}
ct_vit.load_state_dict(sd, strict=False)

# Move to GPU
ct_vit = ct_vit.cuda()
print(" CTViT ready on", next(ct_vit.parameters()).device)


In [ ]:
def extract_ctvit_features(volume_path, max_slices=12, size=480):
    vol = nib.load(volume_path).get_fdata().astype("float32")
    vol = torch.clamp(torch.from_numpy(vol), -1000.0, 400.0)
    vol = (vol + 1000.0) / 1400.0
    x = vol.unsqueeze(0).unsqueeze(0).to(device)
    x = F.interpolate(x, size=(max_slices, size, size),
                      mode="trilinear", align_corners=False)
    with torch.no_grad():
        tokens = ct_vit(x, return_encoded_tokens=True)
    B,T,H,W,D = tokens.shape
    feats = tokens.reshape(B, T*H*W, D).squeeze(0).cpu()
    return feats


Load LLaMA-3 8B with LoRA Adapter

In [ ]:
# Load tokenizer, base, and LoRA adapter
bnb = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    trust_remote_code=True, padding_side="left"
)
tokenizer.pad_token_id = tokenizer.eos_token_id

base = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True
)

model = PeftModel.from_pretrained(
    base,
    "/content/drive/MyDrive/ct_lora_500scans",
    device_map="auto"
)
model.eval()
print(" Loaded LLaMA-8B + LoRA adapter")


In [ ]:
# Load joint ProjectionHead
import torch.nn as nn

class ProjectionHead(nn.Module):
    def __init__(self, in_dim=512, out_dim=4096):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(inplace=True),
            nn.Linear(in_dim, out_dim),
        )
    def forward(self, x):
        pooled = x.mean(dim=1)
        return self.net(pooled)

proj_head = ProjectionHead(in_dim=512, out_dim=4096).to(device)
proj_state = torch.load(
    "/content/drive/MyDrive/projection_head_500scans_joint.pt",
    map_location=device
)
proj_head.load_state_dict(proj_state)
proj_head.eval()
print(" Loaded ProjectionHead")


**Define the Report Generation Function**

-Extracts features from the scan using CTViT

-Projects features via the trained projection head

-Constructs a prompt with findings/impressions format

-Generates a report using the fine-tuned language model

-Returns the final decoded text report

In [ ]:
def generate_report(scan_path, contrast_used=False, scan_region="thorax"):
    Extract & project
    feats = extract_ctvit_features(scan_path).unsqueeze(0).to(device)
    proj  = proj_head(feats)  # [1,4096]

    contrast = "non-contrast" if not contrast_used else "contrast-enhanced"
    prompt = f"""

Generate a CT report with “Findings” and “Impressions”:

"""
    toks     = tokenizer(prompt, return_tensors="pt", padding=True)
    input_ids = toks.input_ids.to(device)
    attn_mask = toks.attention_mask.to(device)

    # Build inputs_embeds
    text_embs   = model.get_input_embeddings()(input_ids)          # [1,L,dim]
    proj_emb    = proj.unsqueeze(1).to(text_embs.dtype)            # [1,1,dim]
    inputs_embs = torch.cat([proj_emb, text_embs], dim=1)          # [1,L+1,dim]
    full_mask   = torch.cat([torch.ones(1,1,device=device), attn_mask], dim=1)

    # Decode with tuned sampling & no-repeat
    out = model.generate(
        inputs_embeds=inputs_embs,
        attention_mask=full_mask,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3,
        eos_token_id=tokenizer.eos_token_id
    )

    # Return the completed report
    return tokenizer.decode(out[0], skip_special_tokens=True)


In [ ]:
from huggingface_hub import hf_hub_download
import pandas as pd

repo_id = "ibrahimhamamci/CT-RATE"
directory_name = "dataset/"

data=pd.read_csv("/content/example_download_script/labels.csv")
count = 0
for name in data["VolumeName"]:
    folder1 = name.split("_")[0]
    folder2 = name.split("_")[1]
    folder = folder1 + "_" + folder2
    folder3 = name.split("_")[2]
    subfolder = folder + "_" + folder3
    subfolder = directory_name + folder + "/" + subfolder

    hf_hub_download(repo_id=repo_id, repo_type="dataset", subfolder=subfolder, filename = name, local_dir = "data_volumes")
    count += 1
    if count == 5:
      break
%cd /content

In [ ]:
# Recursive feature extraction
data_root   = "/content/example_download_script/example_download_script/example_download_script/data_volumes"
feats_list  = []
names_list  = []

for dirpath, _, files in os.walk(data_root):
    for fn in files:
        if not fn.endswith(".nii.gz"):
            continue
        path = os.path.join(dirpath, fn)

        # preprocess volume
        vol = nib.load(path).get_fdata().astype("float32")
        vol = torch.clamp(torch.from_numpy(vol), -1000.0, 400.0)
        vol = (vol + 1000.0) / 1400.0
        x   = vol.unsqueeze(0).unsqueeze(0).cuda()
        x   = F.interpolate(x, size=(12,480,480),
                            mode="trilinear", align_corners=False)

        # forward pass
        with torch.no_grad():
            tokens = ct_vit(x, return_encoded_tokens=True)  # [1,T,H,W,D]
        B, T, H, W, D = tokens.shape
        feats = tokens.reshape(B, T*H*W, D).squeeze(0).cpu()  # [N_patches,512]

        # sanity
        assert not feats.isnan().any(), f"NaN in {fn}"
        assert not feats.isinf().any(), f"Inf in {fn}"
        print(f" {fn} → {feats.shape}")

        feats_list.append(feats)
        names_list.append(fn[:-7])

print(f" Extracted features for {len(feats_list)} volumes")

In [ ]:
import pandas as pd
import os
from huggingface_hub import hf_hub_download

# Download the train_reports.csv
reports_path = hf_hub_download(
    repo_id="ibrahimhamamci/CT-RATE",
    repo_type="dataset",
    subfolder="dataset/radiology_text_reports",
    filename="reports.csv",
    use_auth_token=True
)
print(" CSV downloaded to:", reports_path)

# Load and drop any empty entries
df_reports = pd.read_csv(reports_path).dropna(subset=["Findings_EN", "Impressions_EN"])
print(f" Loaded {len(df_reports)} non‑empty report rows")

# uild lookup dict, stripping “.nii.gz”
report_index = {}
for _, row in df_reports.iterrows():
    name = row.VolumeName
    if name.endswith(".nii.gz"):
        key = name[:-7]
    else:
        key = name
    report_index[key] = {
        "Findings_EN": row.Findings_EN,
        "Impressions_EN": row.Impressions_EN
    }

# Sanity‑check alignment with extracted names_list
missing = [n for n in names_list if n not in report_index]
assert len(missing)==0, f"Reports missing for: {missing[:5]}… ({len(missing)} total)"
print(f" report_index contains all {len(names_list)} extracted volumes")


In [ ]:
refs = []
preds = []

print("Running on:", data_root)
print("="*60)

for name in names_list:
    # Find the full path
    scan_file = None
    for root, _, files in os.walk(data_root):
        target = f"{name}.nii.gz"
        if target in files:
            scan_file = os.path.join(root, target)
            break
    if scan_file is None:
        print(f" Could not find {name}.nii.gz")
        continue

    # Ground truth
    gt = report_index[name]
    ground_truth = gt["Findings_EN"] + " " + gt["Impressions_EN"]

    # Generate output
    try:
        output = generate_report(scan_file)
        if not output.strip():
            print(f" Empty output for {name}")
            continue
    except Exception as e:
        print(f" Error in generating report for {name}: {e}")
        continue

    # Store for metrics
    refs.append(ground_truth.strip())
    preds.append(output.strip())

    print(f"\n {name}")
    print(" Ground Truth:", ground_truth)
    print(" Prediction:", output)
    print("-"*60)

print(f" Stored {len(refs)} reference-prediction pairs")


In [ ]:
# Install the evaluation libraries
!pip install --quiet evaluate nltk rouge-score bert-score


In [ ]:
import evaluate
import nltk
nltk.download('punkt')

# Load metrics
bleu      = evaluate.load("bleu")
rouge     = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

# Check length match
assert len(preds) == len(refs), f"Mismatched lengths: {len(preds)} preds vs {len(refs)} refs"

# Drop empty strings
filtered_pairs = [(p, r) for p, r in zip(preds, refs) if p.strip() and r.strip()]
if len(filtered_pairs) == 0:
    raise ValueError("No valid prediction-reference pairs found!")

filtered_preds, filtered_refs = zip(*filtered_pairs)

# BERTScore (semantic similarity)
bert_res = bertscore.compute(
    predictions=list(filtered_preds),
    references=list(filtered_refs),
    lang="en"
)
f1_score = sum(bert_res["f1"]) / len(bert_res["f1"])
print("BERTScore F1 (avg):", round(f1_score, 4))


**Output**:


BERTScore F1 (avg): 0.8432

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

keywords = ["nodule", "ground-glass", "effusion", "consolidation", "atelectasis"]

# Build binary vectors
y_true, y_pred = [], []
for gt, pred in zip(refs, preds):
    true_vec = [1 if kw in gt.lower() else 0 for kw in keywords]
    pred_vec = [1 if kw in pred.lower() else 0 for kw in keywords]
    y_true.append(true_vec)
    y_pred.append(pred_vec)

# Compute micro‐averaged precision/recall/F1
import numpy as np
y_true = np.array(y_true)
y_pred = np.array(y_pred)

prec, rec, f1, _ = precision_recall_fscore_support(
    y_true, y_pred,
    average="micro",
    zero_division=0
)
print(f"Keyword F1: {f1:.3f} (P={prec:.3f}, R={rec:.3f})")


**Output**:

Keyword F1: 0.700 (P=0.636, R=0.778)




Like I mentioned earlier, CT scans are large and complex, usually around 400 MB each, making it tough to process and generate reports. With limited resources, I did my best to train and test the model on this challenging data.


The F1 score of 0.700 (Precision=0.636, Recall=0.778) and a BERTScore F1 of 0.8432 show which I think is a reasonable performance given these constraints, but also highlight the difficulty of the task. There is still a lot of scope for improvement and further development to achieve better accuracy and efficiency in CT report generation.




# **Acknowledgements:**
The CT visual feature extractor (CT-ViT / CT-CLIP) and the CT-RATE dataset used in this project were created and released by Ibrahim Hamamcı and collaborators. Their weeks-long training on high-performance GPUs and their decision to share the artifacts made this project possible. Original repositories/dataset pages:



CT-CLIP: https://github.com/ibrahimethemhamamci/CT-CLIP


CT-RATE dataset: https://huggingface.co/datasets/ibrahimhamamci/CT-RATE